In [ ]:
from pathlib import Path

EXPERIMENT_DIR = Path.cwd().resolve()
if not (EXPERIMENT_DIR / "experiment.ipynb").is_file():
    candidate = EXPERIMENT_DIR.parent
    if (candidate / "experiment.ipynb").is_file():
        EXPERIMENT_DIR = candidate
if not (EXPERIMENT_DIR / "experiment.ipynb").is_file():
    raise RuntimeError("Start Jupyter in this experiment directory or its notebooks directory.")

GENERATED_DATA_DIR = EXPERIMENT_DIR / "runs" / "canonical-targets" / "generated-data"
TRAINED_MODEL_DIR = EXPERIMENT_DIR / "runs" / "canonical-targets" / "trained-models"
PROJECT_DIR = EXPERIMENT_DIR.parents[3]
EXPERIMENT_ID = EXPERIMENT_DIR.relative_to(PROJECT_DIR / "experiments")
RESULT_DIR = PROJECT_DIR / "figures" / "paper"
for directory in (GENERATED_DATA_DIR, TRAINED_MODEL_DIR, RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Ignore the short initial prediction transient.
PLOT_START_INDEX = 25

def data_path(filename):
    return GENERATED_DATA_DIR / filename

def model_path(filename):
    return TRAINED_MODEL_DIR / filename

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### Generate data

In [ ]:
# Define variables
k = 1            # spring constant
m = 1            # mass 
b = 0.02        # damping coefficient
noise_var = 0.04
omega_0 = np.sqrt(k / m)
beta = b / (2 * m)
w1 = np.sqrt(omega_0**2 - beta**2)  # Damped natural frequency

In [ ]:
def init_conditions(num_samples, energy_range=(0.2,1)):
    energies = np.random.uniform(energy_range[0], energy_range[1], num_samples)
    phis = np.random.uniform(0, 2 * np.pi, num_samples)
    As = []
    
    for E in energies:
        Amp = np.sqrt(2 * E / k)
        As.append(Amp)

    return np.array(As), phis

def position(A, phi, t):
    return A * np.exp(-beta * t) * np.cos(w1 * t + phi)

def linear_momentum(A, phi, t):
    return -A * m * np.exp(-beta * t) * (beta * np.cos(w1 * t + phi) + w1 * np.sin(w1 * t + phi))

def generalized_momentum(A, phi, t):
    return -A * m * np.exp(beta * t) * (beta * np.cos(w1 * t + phi) + w1 * np.sin(w1 * t + phi))

def get_derivative(x_curr, x_prev, time_step):
    return (x_curr - x_prev)/time_step

def compute_derivatives(values : np.ndarray, time_step):
    return np.gradient(values, time_step)

def add_noise(qs, ps):
    return qs + np.random.normal(0, noise_var, len(qs)), ps + np.random.normal(0, noise_var, len(ps))

def generate_damped_oscillator_data(num_trajectories, time_step, noisy=False, num_data=314, sample_all=False, sample_every=100, energy_range=(0.2,1), use_generalized_momentum=False):
    As_0, phis_0 = init_conditions(num_trajectories, energy_range)
    
    qs, ps, ts = [], [], []
    q_dots, p_dots = [], []
    
    for A0, phi_0 in zip(As_0, phis_0):
        t_vals = np.arange(0, num_data, time_step)

        if noisy:
            qs_traj, ps_traj = add_noise(position(A0, phi_0, t_vals), generalized_momentum(A0, phi_0, t_vals) if use_generalized_momentum else linear_momentum(A0, phi_0, t_vals))
        else:
            qs_traj, ps_traj = position(A0, phi_0, t_vals), generalized_momentum(A0, phi_0, t_vals) if use_generalized_momentum else linear_momentum(A0, phi_0, t_vals)
        
        q_dot, p_dot = compute_derivatives(qs_traj, time_step), compute_derivatives(ps_traj, time_step)

        if sample_all: 
            q_dots.append(q_dot)
            p_dots.append(p_dot)
            qs.append(qs_traj)
            ps.append(ps_traj)
            ts.append(t_vals)
        else:        
            # Sample every 100th data 
            q_dots.append(q_dot[::sample_every])
            p_dots.append(p_dot[::sample_every])
            qs.append(qs_traj[::sample_every])
            ps.append(ps_traj[::sample_every])
            ts.append(t_vals[::sample_every])
    
    return np.asarray(qs), np.asarray(ps), np.asarray(q_dots), np.asarray(p_dots), np.asarray(ts)

#### Generate Training Data

In [ ]:
# Parameters
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
num_trajectories = 1200                     # Independent noisy training trajectories
num_data = 314                             # Duration in seconds 
sample_every = 2                           # 1,570 points per trajectory; 1,884,000 total
time_step = 0.1  

# Generate data
qs, ps, q_dots, p_dots, ts = generate_damped_oscillator_data(num_trajectories, time_step, noisy=True, num_data=num_data, sample_every=sample_every, use_generalized_momentum=False)

# Compute canonical momentum
ps_canonical = np.exp((b/m) * ts) * ps
p_dots_canonical = np.exp((b/m) * ts) * (p_dots + (b/m) * ps)

# Reshape into single dataset
qs_flat = qs.reshape(-1)
ps_canonical_flat = ps_canonical.reshape(-1)
ts_flat = ts.reshape(-1)
q_dots_flat = q_dots.reshape(-1)
p_dots_flat = p_dots_canonical.reshape(-1)

# Create DataFrame
train_df = pd.DataFrame({
    "q": qs_flat,
    "p_canonical": ps_canonical_flat,
    "t": ts_flat, 
    "q_dot": q_dots_flat,
    "p_dot": p_dots_flat
})

train_df.to_csv(GENERATED_DATA_DIR / "damped_oscillator_train_noise.csv", index=False)

#### Generate Test Data

In [ ]:
qs_test, ps_test, q_dots_test, p_dots_test, ts_test = generate_damped_oscillator_data(1, time_step, noisy=False, sample_all=True, energy_range=(0.2,0.2), use_generalized_momentum=False)

# Compute canonical momentum
ps_canonical_test = np.exp((b/m) * ts_test) * ps_test
p_dots_canonical_test = np.exp((b/m) * ts_test) * (p_dots_test + (b/m) * ps_test)

qs_test_flat = qs_test.reshape(-1)
ps_canonical_test_flat = ps_canonical_test.reshape(-1)
ts_test_flat = ts_test.reshape(-1)
q_dots_test_flat = q_dots_test.reshape(-1)
p_dots_test_flat = p_dots_canonical_test.reshape(-1)

test_df = pd.DataFrame({
    "q": qs_test_flat, 
    "p_canonical": ps_canonical_test_flat, 
    "t": ts_test_flat, 
    "q_dot": q_dots_test_flat, 
    "p_dot": p_dots_test_flat
})
test_df.to_csv(GENERATED_DATA_DIR / "damped_oscillator_low_energy.csv", index=False)

qs_test, ps_test, q_dots_test, p_dots_test, ts_test = generate_damped_oscillator_data(1, time_step, noisy=False, sample_all=True, energy_range=(1.0, 1.0), use_generalized_momentum=False)

# Compute canonical momentum
ps_canonical_test = np.exp((b/m) * ts_test) * ps_test
p_dots_canonical_test = np.exp((b/m) * ts_test) * (p_dots_test + (b/m) * ps_test)

qs_test_flat = qs_test.reshape(-1)
ps_canonical_test_flat = ps_canonical_test.reshape(-1)
ts_test_flat = ts_test.reshape(-1)
q_dots_test_flat = q_dots_test.reshape(-1)
p_dots_test_flat = p_dots_canonical_test.reshape(-1)

test_df = pd.DataFrame({
    "q": qs_test_flat, 
    "p_canonical": ps_canonical_test_flat, 
    "t": ts_test_flat, 
    "q_dot": q_dots_test_flat, 
    "p_dot": p_dots_test_flat
})
test_df.to_csv(GENERATED_DATA_DIR / "damped_oscillator_high_energy.csv", index=False)

### Load Data

In [ ]:
train_df = pd.read_csv(data_path("damped_oscillator_train_noise.csv"))
test_df_low = pd.read_csv(data_path("damped_oscillator_low_energy.csv"))
test_df_high = pd.read_csv(data_path("damped_oscillator_high_energy.csv"))

### Prepare training data 

In [ ]:
train_df[:5]

In [ ]:
features = train_df.iloc[:, 0:3]
targets = train_df.iloc[:, 3:]

In [ ]:
scaler = MinMaxScaler(feature_range=(-1,1))
# features_scaled = scaler.fit_transform(features)

In [ ]:
# f_tensor = torch.FloatTensor(features_scaled).to(device)
f_tensor = torch.FloatTensor(features.values).to(device)
t_tensor = torch.FloatTensor(targets.values).to(device)

In [ ]:
train_dataset = TensorDataset(f_tensor, t_tensor)
BATCH_SIZE = 2048
train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

### Constants

In [ ]:
HIDDEN_DIM = 256
LEARNING_RATE = 1e-3
MIN_LEARNING_RATE = 1e-5
EPOCHS = 15 
INPUT_DIM = 3
OUTPUT_DIM = 1 
OUTPUT_DIM_B = 2
HIDDEN_LAYERS_DIM = 4 

# Scale features inside forward so HNN derivatives retain the chain rule.
INPUT_OFFSET = (0.0, 0.0, 157.0)
INPUT_SCALE = (1.5, 2.0, 157.0)  # Small p scale keeps dH/dp resolvable


### Model Definitions

In [ ]:
# Build Neural Network
class BaselineNN(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM_B, hidden_layers=HIDDEN_LAYERS_DIM):
        super(BaselineNN, self).__init__()
        
        self.register_buffer("input_offset", torch.tensor(INPUT_OFFSET))
        self.register_buffer("input_scale", torch.tensor(INPUT_SCALE))
        self.inp_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(hidden_layers)])
        self.out_layer = nn.Linear(hidden_dim, output_dim)
        self.activation = torch.tanh

    def forward(self, x):
        x = (x - self.input_offset) / self.input_scale
        x = self.activation(self.inp_layer(x))
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        x = self.out_layer(x)
        return x
        
class HNN(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM, hidden_layers=HIDDEN_LAYERS_DIM):
        super(HNN, self).__init__()

        self.register_buffer("input_offset", torch.tensor(INPUT_OFFSET))
        self.register_buffer("input_scale", torch.tensor(INPUT_SCALE))
        self.inp_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(hidden_layers)])
        self.out_layer = nn.Linear(hidden_dim, output_dim)
        self.activation = torch.tanh

    def forward(self, x):
        x = (x - self.input_offset) / self.input_scale
        x = self.activation(self.inp_layer(x))
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        x = self.out_layer(x)
        return x

    def grad_forward(self, x):
        # Ensure that x requires gradient.
        x.requires_grad_(True)
        H = self.forward(x)
        grad_outputs = torch.ones_like(H)
        
        grad_H = torch.autograd.grad(outputs=H, inputs=x, grad_outputs=grad_outputs, create_graph=True)[0]
        dhdq = grad_H[:,0]
        dhdp = grad_H[:,1]

        dpdt = -dhdq
        dqdt = dhdp
        
        return dpdt, dqdt

In [ ]:
def train_baselineNN(data_dl, learning_rate=LEARNING_RATE, epochs=EPOCHS):
    best_model_state_baseline = None
    min_loss = float('inf')
    model = BaselineNN(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM).to(device)
    
    # SmoothL1 keeps the exp(bt)-amplified late-time noise from dominating the loss.
    criterion = nn.SmoothL1Loss()
    
    # Create an Adam optimizer.
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=MIN_LEARNING_RATE)
    
    model.train()
    
    for epoch in range(epochs):
        epoch_loss = 0.0 
        batch_count = 0

        for inputs, targets in data_dl:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()

            outputs = model(inputs) 
            loss = criterion(outputs, targets)
            loss.backward()  

            optimizer.step()

            epoch_loss += loss.item()
            batch_count += 1

        avg_loss = epoch_loss / max(batch_count, 1)
        
        if epoch % 10 == 0:
            print(f"Epoch: {epoch}/{epochs}, Avg Loss: {avg_loss:.6f}, LR: {optimizer.param_groups[0]['lr']:.2e}")

        if avg_loss < min_loss and batch_count > 0:
            min_loss = avg_loss
            best_model_state_baseline = copy.deepcopy(model.state_dict())

        scheduler.step()

    print(f"Training completed. Best loss: {min_loss:.6f}")
    
    return model, best_model_state_baseline
    
def train_HNN(data_dl, learning_rate=LEARNING_RATE, epochs=EPOCHS):
    best_model_state = None
    min_loss = float('inf')
    model = HNN(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM).to(device)
    
    # SmoothL1 keeps the exp(bt)-amplified late-time noise from dominating the loss.
    criterion = nn.SmoothL1Loss()
    
    # Create an Adam optimizer.
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=MIN_LEARNING_RATE)
    
    model.train()
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        batch_count = 0

        for inputs, targets in data_dl:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            
            dpdt, dqdt = model.grad_forward(inputs)
            predictions = torch.stack([dqdt, dpdt], dim=1) 
            
            loss = criterion(predictions, targets)
            loss.backward()
            
            optimizer.step()

            epoch_loss += loss.item()
            batch_count += 1

        avg_loss = epoch_loss / max(batch_count, 1)

        if epoch % 10 == 0:
            print(f"Epoch: {epoch}/{epochs}, Avg Loss: {avg_loss:.6f}, LR: {optimizer.param_groups[0]['lr']:.2e}")

        if avg_loss < min_loss and batch_count > 0:
            min_loss = avg_loss
            best_model_state = copy.deepcopy(model.state_dict())

        scheduler.step()
            
    print(f"Training completed. Best loss: {min_loss:.6f}")

    return model, best_model_state

### Train model

In [ ]:
torch.manual_seed(RANDOM_SEED)
_, best_model_state_baseline = train_baselineNN(train_dl, epochs=EPOCHS)

In [ ]:
torch.manual_seed(RANDOM_SEED)
_, best_model_state = train_HNN(train_dl, epochs=EPOCHS)

In [ ]:
# Save the best model state for baseline if it was updated
if best_model_state_baseline is not None: 
    torch.save(best_model_state_baseline, TRAINED_MODEL_DIR / "best_model_baseline_noise.pth")

# Load the best model 
best_model_baseline = BaselineNN().to(device)
best_model_baseline.load_state_dict(torch.load(model_path("best_model_baseline_noise.pth"), map_location="cpu"), strict=False)

In [ ]:
# Save the best model state if it was updated
if best_model_state is not None: 
    torch.save(best_model_state, TRAINED_MODEL_DIR / "best_model_hnn_noise.pth")

In [ ]:
# Load the best model 
best_model_hnn = HNN().to(device)
best_model_hnn.load_state_dict(torch.load(model_path("best_model_hnn_noise.pth"), map_location="cpu"), strict=False)

### Prepare testing data 

In [ ]:
features_test = test_df_high.iloc[:, 0:3]
targets_test = test_df_high.iloc[:, 3:]

# features_test_scaled = scaler.fit_transform(features_test)
# f_tensor_test = torch.FloatTensor(features_test_scaled).to(device)
f_tensor_test = torch.FloatTensor(features_test.values).to(device)
t_tensor_test = torch.FloatTensor(targets_test.values).to(device)

test_dataset = TensorDataset(f_tensor_test, t_tensor_test)
test_dl_high = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

In [ ]:
features_test_l = test_df_low.iloc[:, 0:3]
targets_test_l = test_df_low.iloc[:, 3:]

# features_test_scaled_l = scaler.fit_transform(features_test_l)
# f_tensor_test_l = torch.FloatTensor(features_test_scaled_l).to(device)
f_tensor_test_l = torch.FloatTensor(features_test_l.values).to(device)
t_tensor_test_l = torch.FloatTensor(targets_test_l.values).to(device)

test_dataset_l = TensorDataset(f_tensor_test_l, t_tensor_test_l)
test_dl_low = DataLoader(test_dataset_l, batch_size=len(test_dataset_l), shuffle=False)

### Plot results

In [ ]:
def evaluate_model_baseline(model, data_dl):
    model.eval()
    predictions = []
    true = []

    for test_inps, test_targets in data_dl: 
        test_inps, test_targets = test_inps.to(device), test_targets.to(device)
        outputs = model(test_inps)
        pred_batch = outputs.detach().cpu().numpy()
        true_batch = test_targets.detach().cpu().numpy()

        predictions.append(pred_batch)
        true.append(true_batch)

    y_true = np.concatenate(true, axis=0) 
    y_pred = np.concatenate(predictions, axis=0)  

    return y_pred, y_true

def evaluate_model(model, data_dl):
    model.eval()
    predictions = []
    true = []
    Hs = []

    for test_inps, test_targets in data_dl: 
        test_inps, test_targets = test_inps.to(device), test_targets.to(device)
        H = model(test_inps).detach().cpu().numpy()
        dpdt, dqdt = model.grad_forward(test_inps)
        pred_batch = torch.stack([dqdt, dpdt], dim=1).detach().cpu().numpy()
        true_batch = test_targets.detach().cpu().numpy()

        predictions.append(pred_batch)
        true.append(true_batch)
        Hs.append(H)

    y_true = np.concatenate(true, axis=0) 
    y_pred = np.concatenate(predictions, axis=0)  
    Hs = np.concatenate(Hs, axis=0)

    return y_pred, y_true, Hs

In [ ]:
y_pred_base_high, y_true_base_high = evaluate_model_baseline(best_model_baseline, test_dl_high)
y_pred_base_low, y_true_base_low = evaluate_model_baseline(best_model_baseline, test_dl_low)

y_pred_hnn_high, y_true_hnn_high, H_high = evaluate_model(best_model_hnn, test_dl_high)
y_pred_hnn_low, y_true_hnn_low, H_low = evaluate_model(best_model_hnn, test_dl_low)

In [ ]:
# plt.plot(np.arange(len(y_pred_hnn_low)), H_low)

In [ ]:
def plot_phase_space_dot(y_true, y_pred, df, save_path=None, save=False):
    q_dot_true = y_true[:, 0]
    p_canonical_dot_true = y_true[:, 1]
    q_dot_pred = y_pred[:, 0]
    p_canonical_dot_pred = y_pred[:, 1]
    t = df["t"][:1000]
    
    # Both targets and predictions are canonical derivatives; convert to linear momentum derivatives.
    p_canonical = df["p_canonical"][:1000]
    p_dot_true = np.exp(-(b/m) * t) * (p_canonical_dot_true - (b/m) * p_canonical)
    p_dot_pred = np.exp(-(b/m) * t) * (p_canonical_dot_pred - (b/m) * p_canonical)

    plt.figure(figsize=(7, 5))
    plt.plot(q_dot_true, p_dot_true, label='True', color='red')
    plt.plot(q_dot_pred, p_dot_pred, alpha=0.6, label='Predicted', color='blue', linestyle='--')
    plt.xlabel(r'$\dot{q}$', fontsize=14, fontweight='bold')
    plt.ylabel(r'$\dot{p}$', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10, frameon=False)

    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()

    if save_path and save:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

In [ ]:
# plot_phase_space_dot(y_true_base_low[PLOT_START_INDEX:1000], y_pred_base_low[PLOT_START_INDEX:1000], test_df_low.iloc[PLOT_START_INDEX:1000].reset_index(drop=True))

In [ ]:
plot_phase_space_dot(y_true_hnn_low[PLOT_START_INDEX:1000], y_pred_hnn_low[PLOT_START_INDEX:1000], test_df_low.iloc[PLOT_START_INDEX:1000].reset_index(drop=True), save_path=RESULT_DIR / "f5d.png", save=True)

In [ ]:
# plot_phase_space_dot(y_true_base_high[PLOT_START_INDEX:1000], y_pred_base_high[PLOT_START_INDEX:1000], test_df_high.iloc[PLOT_START_INDEX:1000].reset_index(drop=True))

In [ ]:
# plot_phase_space_dot(y_true_hnn_high[PLOT_START_INDEX:1000], y_pred_hnn_high[PLOT_START_INDEX:1000], test_df_high.iloc[PLOT_START_INDEX:1000].reset_index(drop=True))

In [ ]:
def plot_qdot_time(y_true, y_pred, plot_true=True, plot_pred=True, n=250, save_path=None, save=False):
    q_dot_true = y_true[:, 0]
    q_dot_pred = y_pred[:, 0]
    plt.figure(figsize=(7, 5))
    
    if plot_true:
        plt.plot(q_dot_true[:n], label="True", color="red")

    if plot_pred:
        plt.plot(q_dot_pred[:n], label="Predicted", color="blue", linestyle="--")
        
    plt.xlabel('t', fontsize=14, fontweight='bold')
    plt.ylabel(r'$\dot{q}$', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10, frameon=False)

    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()

    if save_path and save:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

In [ ]:
# baseline low energy
# plot_qdot_time(y_true_base_low[PLOT_START_INDEX:], y_pred_base_low[PLOT_START_INDEX:], n=725)

In [ ]:
# hnn low energy
plot_qdot_time(y_true_hnn_low[PLOT_START_INDEX:], y_pred_hnn_low[PLOT_START_INDEX:], n=725, save_path=RESULT_DIR / "f5b.png", save=True)

In [ ]:
# baseline high energy
# plot_qdot_time(y_true_base_high[PLOT_START_INDEX:], y_pred_base_high[PLOT_START_INDEX:], n=725)

In [ ]:
# hnn high energy
# plot_qdot_time(y_true_hnn_high[PLOT_START_INDEX:], y_pred_hnn_high[PLOT_START_INDEX:], n=725)

In [ ]:
def plot_hamiltonian_high(test_df, H_pred, plot_true=True, plot_pred=True, n=250, save_path=None, save=False):
    q = test_df["q"].values
    p_canonical = test_df["p_canonical"].values
    q_dot = test_df["q_dot"].values
    t = test_df["t"]
   
    # H_true = np.exp((b/m)*t) * 0.5 *(m * q_dot**2 + k * q**2)
    H_true = np.exp((-b/m) * t) * (p_canonical**2/(2*m)) + np.exp((b/m) * t) * ((k * q**2)/2)
    H_mean = np.mean(H_true)
    H_std = np.std(H_true)
    H_pred_scaled = (H_pred - np.mean(H_pred)) / np.std(H_pred) * H_std + H_mean 

    plt.figure(figsize=(7, 5))
    if plot_true:
        plt.plot(H_true[:n], label="True", color="red")

    if plot_pred:
        plt.plot(H_pred_scaled[:n], label="Predicted", color="blue", linestyle="--")
        
    plt.xlabel('t', fontsize=14, fontweight='bold')
    plt.ylabel('Hamiltonian', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10, frameon=False)

    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.title("Eqn 31 - unscaled data")

    plt.tight_layout()

    if save_path and save:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

In [ ]:
# plot_hamiltonian_high(test_df_high.iloc[PLOT_START_INDEX:].reset_index(drop=True), H_high[PLOT_START_INDEX:], n=len(H_high) - PLOT_START_INDEX)

In [ ]:
def plot_hamiltonian_low(test_df, H_pred, y_pred, plot_true=True, plot_pred=True, n=250, save_path=None, save=False):
    q = test_df["q"].values
    p_canonical = test_df["p_canonical"].values
    q_dot = test_df["q_dot"].values
    t = test_df["t"]
    
    # H_true = np.exp((b/m)*t) * 0.5 *(m * q_dot**2 + k * q**2)
    H_true = np.exp((-b/m) * t) * (p_canonical**2/(2*m)) + np.exp((b/m) * t) * ((k * q**2)/2)
    H_mean = np.mean(H_true)
    H_std = np.std(H_true)
    H_pred_scaled = (H_pred - np.mean(H_pred)) / np.std(H_pred) * H_std + H_mean
    plt.figure(figsize=(7, 5))
    
    if plot_true:
        plt.plot(H_true[:n], label="True", color="red")

    if plot_pred:
        plt.plot(H_pred_scaled[:n], label="Predicted", color="blue", linestyle="--")
        
    plt.xlabel('t', fontsize=14, fontweight='bold')
    plt.ylabel('Hamiltonian', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10, frameon=False)

    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.ylim((0.125, 0.275))

    plt.tight_layout()

    if save_path and save:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

In [ ]:
plot_hamiltonian_low(test_df_low.iloc[PLOT_START_INDEX:].reset_index(drop=True), H_low[PLOT_START_INDEX:], y_pred_hnn_low[PLOT_START_INDEX:], n=len(H_low) - PLOT_START_INDEX, save_path=RESULT_DIR / "f5f.png", save=True)